In [ ]:
from bisect import bisect_left, bisect_right
from collections import deque
from math import inf


def min_interfered_posts(posts, L, candidates, r=250):
    P = sorted(q % L for q in posts)
    n = len(P)
    if n == 0:
        return 0
    if not candidates:
        return None
    if 2 * r >= L:                      # every candidate covers every post
        return 0

    big = [q - L for q in P] + P + [q + L for q in P]   # three unrolled copies
    arcs = set()
    for c in candidates:
        c %= L
        lo = bisect_left(big, c - r)
        hi = bisect_right(big, c + r) - 1
        if lo > hi:
            continue                    # covers no post
        s, e = lo - n, hi - n
        shift = (s // n) * n            # normalise so that 0 <= s < n
        s, e = s - shift, e - shift
        if e - s + 1 >= n:              # one transmitter covers all posts
            return 0
        arcs.add((s, e))
    arcs = sorted(arcs)
    M = len(arcs)
    if M == 0:
        return None
    S = [a[0] for a in arcs]
    E = [a[1] for a in arcs]

    sg, eg = [0] * M, [0] * M           # first index of same-s / same-e group
    for i in range(1, M):
        sg[i] = sg[i - 1] if S[i] == S[i - 1] else i
        eg[i] = eg[i - 1] if E[i] == E[i - 1] else i
    lo_idx = [bisect_left(E, S[i] - 1) for i in range(M)]
    hi_idx = [min(sg[i], eg[i]) - 1 for i in range(M)]

    best = inf
    for a in range(M):                  # a = chosen arc with smallest start
        sa = S[a]
        dp = [inf] * M
        dp[a] = 0
        dq = deque()
        ptr = a
        for i in range(a + 1, M):
            if S[i] == sa:
                continue
            while ptr <= hi_idx[i]:
                if dp[ptr] < inf:
                    v = dp[ptr] + E[ptr]
                    while dq and dq[-1][0] >= v:
                        dq.pop()
                    dq.append((v, ptr))
                ptr += 1
            lo = max(a, lo_idx[i])
            while dq and dq[0][1] < lo:
                dq.popleft()
            if dq:
                dp[i] = dq[0][0] - S[i] + 1
        for i in range(a, M):
            if dp[i] < inf and E[i] >= sa + n - 1:
                best = min(best, dp[i] + max(0, E[i] - sa - n + 1))
    return None if best == inf else best

In [6]:
posts = [100, 400, 700, 900]
L = 1000
candidates = [50, 350, 650, 950]
min_interfered_posts(posts, L, candidates, r=250)

1

In [ ]:
from bisect import bisect_left, bisect_right
from collections import deque
from math import inf


def min_interfered_posts_with_locations(posts, L, candidates, r=250):
    P = sorted(q % L for q in posts)
    n = len(P)
    if n == 0:
        return 0, []
    if not candidates:
        return None
    if 2 * r >= L:
        return 0, [candidates[0]]

    big = [q - L for q in P] + P + [q + L for q in P]
    where = {}                                  # arc -> a candidate location
    for c0 in candidates:
        c = c0 % L
        lo = bisect_left(big, c - r)
        hi = bisect_right(big, c + r) - 1
        if lo > hi:
            continue
        s, e = lo - n, hi - n
        shift = (s // n) * n
        s, e = s - shift, e - shift
        if e - s + 1 >= n:
            return 0, [c0]
        where.setdefault((s, e), c0)
    arcs = sorted(where)
    M = len(arcs)
    if M == 0:
        return None
    S = [a[0] for a in arcs]
    E = [a[1] for a in arcs]

    sg, eg = [0] * M, [0] * M
    for i in range(1, M):
        sg[i] = sg[i - 1] if S[i] == S[i - 1] else i
        eg[i] = eg[i - 1] if E[i] == E[i - 1] else i
    lo_idx = [bisect_left(E, S[i] - 1) for i in range(M)]
    hi_idx = [min(sg[i], eg[i]) - 1 for i in range(M)]

    best, best_chain = inf, None
    for a in range(M):
        sa = S[a]
        dp = [inf] * M
        par = [-1] * M
        dp[a] = 0
        dq = deque()
        ptr = a
        for i in range(a + 1, M):
            if S[i] == sa:
                continue
            while ptr <= hi_idx[i]:
                if dp[ptr] < inf:
                    v = dp[ptr] + E[ptr]
                    while dq and dq[-1][0] >= v:
                        dq.pop()
                    dq.append((v, ptr))
                ptr += 1
            lo = max(a, lo_idx[i])
            while dq and dq[0][1] < lo:
                dq.popleft()
            if dq:
                dp[i] = dq[0][0] - S[i] + 1
                par[i] = dq[0][1]
        for i in range(a, M):
            if dp[i] < inf and E[i] >= sa + n - 1:
                tot = dp[i] + max(0, E[i] - sa - n + 1)
                if tot < best:
                    chain, k = [], i            # follow parent pointers
                    while k != -1:
                        chain.append(k)
                        k = par[k]
                    best, best_chain = tot, chain[::-1]
    if best == inf:
        return None
    return best, [where[arcs[k]] for k in best_chain]

In [8]:
posts = [100, 400, 700, 900]
L = 1000
candidates = [50, 350, 650, 950]
min_interfered_posts_with_locations(posts, L, candidates, r=250)

(1, [350, 650])